In [1]:
import os
os.chdir("../dgl_ptm/")

import sys
import zarr
import dgl_ptm
import networkx as nx

os.environ["DGLBACKEND"] = "pytorch"
import torch

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

In [2]:
model = dgl_ptm.SVEIRModel(model_identifier='seir_test')
seed = 42

In [3]:
model.set_model_parameters(**{
    'number_agents': 200,
    'seed':seed,
    'spatial': True,
    'spatial_creation_args':{
        "method": "custom_import",
        "path": "ghana_grid.npy",
        "properties": {"population": 0}
    },
    'spatial_assignment_args':{'method':"property",'property':"population"},
    'initial_graph_type': 'barabasi-albert',
    'initial_graph_args': {'seed': seed, 'new_node_edges':5},
    'device': 'cpu',
    'step_target':5,
    'steering_parameters': {
        'npath':'./agent_data.zarr',
        'epath':'./edge_data',
        'ndata':[['compartments'],['initial_only',[]]],
        'edata':None,
        'mode':'w',
        'proximity_decay_rate':0.05,
        'truncation_weight':1e-10,
        'infection_probability': 0.05,
        'recovery_rate': 0.1,
        'vaccination_rate':0.01,
        'vaccine_efficacy':0.9,
        'exposure_period':5,
        'initial_infected_proportion':0.03,
        'step_type':'seir'
    }
})

In [4]:
model.initialize_model()
print()
print(model.config)

Model torch seed set to 42
Using seed 42 for network creation with 5 edges requested.
200 agents initialized on cpu device

model_identifier='seir_test' description='' device='cpu' seed=42 number_agents=200 spatial=True spatial_creation_args=GridCreationParams(method='custom_import', x=10, y=10, properties={'population': 0}, path='ghana_grid.npy') spatial_assignment_args=GridAssignmentParams(method='property', property='population', path=None) initial_graph_type='barabasi-albert' initial_graph_args=InitialGraphArgs(seed=42, new_node_edges=5) step_target=5 steering_parameters=SteeringParamsSVEIR(npath='seir_test/agent_data.zarr', epath='seir_test/edge_data', ndata=[['compartments'], ['initial_only', []]], edata=None, mode='w', infection_probability=0.05, recovery_rate=0.1, vaccination_rate=0.01, vaccine_efficacy=0.9, exposure_period=5, initial_infected_proportion=0.03, truncation_weight=1e-10, proximity_decay_rate=0.05, step_type='seir', data_collection_period=1, data_collection_list=No

In [5]:
# dist = []
# u, v = model.graph.edges()
# x_u = model.graph.ndata['x'][u]
# y_u = model.graph.ndata['y'][u]
# x_v = model.graph.ndata['x'][v]
# y_v = model.graph.ndata['y'][v]
# distance = torch.sqrt((x_u - x_v)**2 + (y_u - y_v)**2)
# plt.scatter(distance, model.graph.edata["weight"])
# plt.show()

In [6]:
# nx.draw(model.graph.to_networkx())

In [7]:
# plt.imshow(model.grid_environment.get_slice("population"), norm=LogNorm())
# plt.gca().invert_yaxis()
# plt.scatter(model.graph.ndata['x'], model.graph.ndata['y'], c="red", s=10, alpha=0.3)
# plt.show()

In [8]:
model.run()

performing step 0 of 5
ATTENTION: No edge data collection requested for this simulation!
performing step 1 of 5
performing step 2 of 5
performing step 3 of 5
performing step 4 of 5


### Results

In [9]:
compartment_data = zarr.load('./seir_test/agent_data.zarr/compartments')